# Quantise during training (quantisation-aware training)

Instead of training in the reals and rounding afterwards (see `quantise_after_training`), the
network trains **through** the rounding: its forward pass fake-quantises the weights, so the
integer weights we verify are exactly the ones training optimised — the certificate holds *by
construction*, with no post-hoc rounding gap.

The program, training loop, `quantise` step, and LIA `verify` live in `_ranking.py`.

In [ ]:
import torch
import sys
from pathlib import Path

for _d in (Path.cwd(), Path.cwd() / "tutorials"):
    if (_d / "_ranking.py").exists():
        sys.path.insert(0, str(_d)); break
import _ranking as rk

## The integer program

The same integer counter as the PTQ tutorial (`sugar.Module`, theory `LIA`): `x` increments
while `x < y` or `x < z`, resets otherwise; `y`, `z` are constant targets.

In [ ]:
program, state_vars = rk.make_program()
print(program.with_varnames(dict(zip(state_vars, "xyz"))))

module
  interface
    x : Int([1,1])
    y : Int([1,1])
    z : Int([1,1])
  atom controls x, y, z reads x, y, z
  init
    #9 := [[0]] 
    X(x) := Id #9
    #10 := [[0]] 
    X(y) := Id #10
    #11 := [[0]] 
    X(z) := Id #11
  delay
    d(x) := ZERO 
    d(y) := ZERO 
    d(z) := ZERO 
  update
    #12 := Lt (x, y)
    #13 := Lt (x, z)
    #14 := Or (#12, #13)
    #15 := [[1]] 
    #16 := Add (x, #15)
    #17 := [[0]] 
    #18 := Ite (#14, #16, #17)
    X(x) := Id #18
    X(y) := Id y
    X(z) := Id z



## The quantisation-aware network

`QATRanking` is the same `[3] -> 4 -> [1]` ReLU net, but its `forward` **fake-quantises** the
weights at a fixed `scale`:

```python
def _fake_quant(w, scale):
    return _RoundSTE.apply(w * scale) / scale     # round in forward, identity gradient (STE)
```

So during training the net computes with the *rounded* weights (the straight-through estimator
lets gradients still flow to the underlying floats). It learns weights that round cleanly at
this `scale`.

In [ ]:
scale = 4
torch.manual_seed(0)
qat_V = rk.Module(rk.QATRanking(scale), combinatorial=True)     # forward fake-quantises at `scale`
rk.train(qat_V, margin=1.0)
print("trained (quantisation-aware)")

trained (quantisation-aware)


## Verify in LIA

`quantise(qat_V, scale)` produces the integer (LIA) module with weights `round(scale * w)` —
which is exactly what the fake-quant forward used during training. We verify with the scaled
margin `delta' = scale * delta`.

In [ ]:
int_V = rk.quantise(qat_V, scale)
print(f"scale={scale}  verify (V>=0, decreases):",
      rk.verify(program, int_V, state_vars, delta=1 * scale))

scale=4  verify (V>=0, decreases): (True, True)


## Result

The integer certificate verifies, and — unlike post-training quantisation — the net we verify
is exactly the one training optimised: the rounding was part of the forward pass, so there is
no post-hoc gap to break the margin.

On a certificate this simple, plain post-training quantisation (with a large enough scale) also
works; quantisation-aware training is the principled choice when quantisation is aggressive or
the decrease margin is tight relative to the rounding error, where post-hoc rounding would
otherwise break the certificate.